# 05 — Backbone lab (experimental)

> **Experimental / internal architecture proof — not part of the public product promise.**

This notebook uses [`examples/backbone-lab`](../backbone-lab): intent YAML (metrics +
queries), RBAC policy rules, consumer-lane emit envelopes, and DuckDB query execution
on fixture seed data.

Set `CM_EXPERIMENTAL=1` when using **CLI** lab commands. The Python API below calls
emitters and runtime directly.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["CM_EXPERIMENTAL"] = "1"

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import backbone_lab_project

PROJECT_DIR = backbone_lab_project()
IDENTITY = "analyst"
print(f"project: {PROJECT_DIR}")

In [ ]:
from clearmetric.compiler import compile
from clearmetric.emitters import emit_compile
from clearmetric.emitters.registry import LAB_COMPILE_FORMATS

compiled = compile(PROJECT_DIR)
node_ids = {node.id for node in compiled.artifact.nodes}
print("lab compile formats:", LAB_COMPILE_FORMATS)
print("has metric:", "metric:executive_revenue" in node_ids)
print("has query:", "query:executive_revenue" in node_ids)

## Admin vs consumer catalog

Admin `catalog` is raw JSON. `consumer-catalog` wraps a policy-projected slice in an
envelope `{format, version, identity, node_count, edge_count, payload}`.

In [ ]:
admin_catalog = json.loads(emit_compile("catalog", compiled))
consumer_raw = emit_compile("consumer-catalog", compiled, identity=IDENTITY)
consumer = json.loads(consumer_raw)

admin_ids = {n["id"] for n in admin_catalog["nodes"]}
consumer_ids = {n["id"] for n in consumer["payload"]["nodes"]}

print(f"admin catalog nodes: {len(admin_ids)}")
print(f"consumer catalog nodes: {len(consumer_ids)}")
print(f"metrics/queries in consumer only: {sorted(consumer_ids - admin_ids)}")

## Frontend contract and AI context

Different consumer formats slice different node kinds — same graph, different views.

In [ ]:
contracts = json.loads(emit_compile("frontend-contract", compiled, identity=IDENTITY))
ai_context = json.loads(emit_compile("ai-context", compiled, identity=IDENTITY))

query = contracts["payload"]["queries"][0]
print(f"frontend-contract query id: {query['id']}")
print(f"SQL preview: {query['sql'][:80]}...")

ai_ids = {n["id"] for n in ai_context["payload"]["nodes"]}
print(f"ai-context kinds: {sorted({n.split(':')[0] for n in ai_ids})}")

## Identity-gated impact

With `identity=`, impact gates the selection (`require_allow`) and filters
`related_ids` to allow-only nodes.

In [ ]:
from clearmetric.compiler.impact import impact

_, ungated = impact(
    PROJECT_DIR,
    selection="orders.amount",
    direction="upstream",
)
_, gated = impact(
    PROJECT_DIR,
    selection="orders.amount",
    direction="upstream",
    identity=IDENTITY,
)

print(f"ungated related: {len(ungated.related_ids)}")
print(f"gated related:   {len(gated.related_ids)}")

## Execute a query contract (DuckDB + fixture seed)

Requires `pip install 'clearmetric-core[runtime]'` (DuckDB). Policy is enforced
before SQL runs — no raw fallback.

In [ ]:
from clearmetric.runtime import execute_project_query

rows = execute_project_query(
    compiled.artifact,
    identity=IDENTITY,
    rules_path=compiled.project.policy.rules,
    query_selection="query:executive_revenue",
    project_dir=PROJECT_DIR,
)
print(rows)

## CLI lab commands (optional)

Same flows via subprocess — requires `CM_EXPERIMENTAL=1` in the environment.

In [ ]:
from clearmetric.cli.runner import run_cm

query_cli = run_cm(
    PROJECT_DIR,
    "query",
    "--identity",
    IDENTITY,
    "query:executive_revenue",
    experimental=True,
)
assert query_cli.returncode == 0, query_cli.stderr
print("CLI query result:", json.loads(query_cli.stdout))